# 10 — Model Selection and Validation
### The Test Set Is Used Once

---

> **Position in the series:**
> `01_statistical_learning_theory` → `02_pac_and_vc_dimensions` → `03_linear_algebra_and_geometry` → `04_pca` → `05_gaussian_mixture_models` → `06_linear_regression` → `07_logistic_regression` → `08_svm_and_kernel_methods` → `09_decision_trees_random_forest_boosting` → **`10_model_selection_and_validation`**

---

## References

| # | Source |
|---|--------|
| [1] | T. Hastie, R. Tibshirani & J. Friedman — *The Elements of Statistical Learning*, 2nd ed., Springer, 2009 — §7.10 (Cross-Validation), §7.10.2 (The Wrong and Right Way to Do Cross-validation) |
| [2] | S. Shalev-Shwartz & S. Ben-David — *Understanding Machine Learning*, Cambridge, 2014 — Ch. 11 (Model Selection and Validation) |
| [3] | A. Géron — *Hands-On Machine Learning with Scikit-Learn, Keras & TensorFlow*, O'Reilly — Ch. 1 (Testing and Validating), Ch. 2 (End-to-End Machine Learning Project) |
| [4] | G. C. Cawley & N. L. C. Talbot — *On Over-fitting in Model Selection and Subsequent Selection Bias in Performance Evaluation*, JMLR, 11, 2010 |
| [5] | scikit-learn documentation — *Cross-validation: evaluating estimator performance*, https://scikit-learn.org/stable/modules/cross_validation.html; *Common pitfalls and recommended practices*, https://scikit-learn.org/stable/common_pitfalls.html |

---

## Structure

This closing notebook has no new model and no new theorem. Each section is a **problem** met when training and reporting a model, followed by its **solution**.

0. The Rule in One Page
1. Problem: The Training Score Flatters the Model
2. Problem: Tuning on the Test Set
3. Problem: One Validation Split Is Noisy
4. Problem: The Preprocessing Saw the Validation Data
5. Problem: The Winner's Validation Score Is Optimistic Too
6. The Whole Procedure, From Data to the Reported Number
7. Checklist and Summary

The models come from `07`, `08` and `09`; here they are taken from scikit-learn, because the subject is the procedure around them.

---
## 0. The Rule in One Page

> **The test set is used once:** after every decision has been made, to compute the number that is reported. Nothing is changed because of that number.

A score estimates the performance on new data only if the data it is computed on played **no role in building the model**. Building the model includes fitting the parameters, and also every choice made by looking at a score: hyperparameters, features, preprocessing, the model family, when to stop.

So the data get three roles, and each decision uses only its own part:

| Part | Used for | How often |
|---|---|---|
| **Training** | fitting the parameters | as often as needed |
| **Validation** (carved out of the training part, by K-fold) | choosing between candidates | as often as needed |
| **Test** | the reported number | **once** |

**The procedure.** The rest of the notebook follows this list.

1. Split the data once into training and test. Lock the test part away.
2. Using the training part only, compare the candidates by K-fold cross-validation, with all the preprocessing inside the folds.
3. Choose with a rule fixed in advance.
4. Refit the chosen candidate on the whole training part.
5. Open the test set, compute the score, report it. Stop.

The cell below does step 1 on the breast cancer data used in `09`. To make the rule concrete, the test part goes into a small object, `LockedTestSet`, that hands the data out one time and raises an error at the second request.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

BLUE, RED, TEAL, ORANGE, PURPLE = '#457B9D', '#E63946', '#2A9D8F', '#F4A261', '#6A0572'

class LockedTestSet:
    # Hands the test data out exactly once.
    def __init__(self, X, y):
        self._X, self._y, self.opened = X, y, False
    def __len__(self):
        return len(self._y)
    def open(self):
        if self.opened:
            raise RuntimeError("The test set has already been used once. To recompute, restart and run the notebook from the top.")
        self.opened = True
        return self._X, self._y

# Step 1: split once (stratified, 80/20) and lock the test part away
X_all, y_all = load_breast_cancer(return_X_y=True)
X_train, X_held, y_train, y_held = train_test_split(X_all, y_all, test_size=0.2, stratify=y_all, random_state=0)
TEST = LockedTestSet(X_held, y_held)
del X_all, y_all, X_held, y_held                    # from here on, only X_train / y_train are visible

print(f"training part: {len(y_train)} points, {X_train.shape[1]} features, fraction of class 1 = {y_train.mean():.3f}")
print(f"test part    : {len(TEST)} points, locked (opened: {TEST.opened})")

---
## 1. Problem: The Training Score Flatters the Model

**Problem.** The model was fitted to make the training score high, so that score says little about new data. A flexible model can reach 100% on the training points by memorizing them (`01`, §4.3).

**Solution.** Score the model on points it was not fitted on. The test set is locked, so we set aside a part of the *training* data: a **validation set**.

**In the cell.** Three models are fitted on 75% of the training part and scored on the remaining 25%. The first is an RBF SVM (`08`, §17) on unscaled features with a kernel far too narrow for them.

In [ ]:
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.25, stratify=y_train, random_state=0)

models = {
    'RBF SVM, unscaled, gamma = 0.01': SVC(C=10, gamma=0.01),
    'decision tree, fully grown'     : DecisionTreeClassifier(random_state=0),
    'RBF SVM, scaled, default gamma' : make_pipeline(StandardScaler(), SVC(C=1)),
}
print(f"{'model':<34} {'fitted on':>10} {'training acc':>13} {'validation acc':>15}")
for name, model in models.items():
    model.fit(X_fit, y_fit)
    print(f"{name:<34} {len(y_fit):>10} {model.score(X_fit, y_fit):>13.3f} {model.score(X_val, y_val):>15.3f}")
print(f"\nalways predicting the majority class: validation acc = {max(y_val.mean(), 1 - y_val.mean()):.3f}")

---
## 2. Problem: Tuning on the Test Set

**Problem.** It is tempting to try many models, score each one on the test set and keep the best. The best of many scores is partly luck, and the luck is reported as if it were skill. Each look is harmless alone; the *selection* is what breaks the estimate.

**How large is the effect?** The experiment removes all the skill: the labels are fair coin flips, independent of the features, so **every** model has true accuracy 0.5. We train $k$ small trees on random subsets of the features, score all of them on a test set of $n=200$ points, and keep the best. The best of $k$ test scores exceeds the truth by roughly $\sqrt{\ln k/(2n)}$, the term that the union bound over $k$ hypotheses produces in `02`, §2.3 (dashed curve in the figure).

**Solution.** Take every decision on validation data. Then the test score is computed for a single model that was fixed beforehand, and it is an honest estimate.

In [ ]:
rng = np.random.default_rng(0)
n, d, k_max = 200, 50, 500
coin = lambda size: (rng.normal(size=(size, d)), rng.integers(0, 2, size))    # labels independent of the features
(X_a, y_a), (X_b, y_b), (X_new, y_new) = coin(n), coin(n), coin(20_000)          # training, "test", fresh data

acc_test, acc_new = [], []
for s in range(k_max):
    feats = rng.choice(d, 5, replace=False)
    tree = DecisionTreeClassifier(max_depth=3, random_state=s).fit(X_a[:, feats], y_a)
    acc_test.append(tree.score(X_b[:, feats], y_b)); acc_new.append(tree.score(X_new[:, feats], y_new))
acc_test, acc_new = np.array(acc_test), np.array(acc_new)

ks = np.arange(1, k_max + 1)
best_so_far = np.array([np.argmax(acc_test[:k]) for k in ks])                 # the model we would keep after k looks
print("coin-flip labels: the true accuracy of every model is 0.500\n")
print(f"{'models tried on the test set':>29} {'best test acc (reported)':>25} {'same model on fresh data':>25}")
for k in [1, 10, 50, 100, 500]:
    b = best_so_far[k - 1]
    print(f"{k:>29} {acc_test[b]:>25.3f} {acc_new[b]:>25.3f}")

fig, ax = plt.subplots(figsize=(8, 4.4))
ax.semilogx(ks, acc_test[best_so_far], color=RED, lw=2.5, label='best score on the test set (what would be reported)')
ax.semilogx(ks, acc_new[best_so_far], color=TEAL, lw=2.5, label='the same model on 20,000 fresh points')
ax.semilogx(ks, 0.5 + np.sqrt(np.log(ks) / (2 * n)), color='gray', lw=1.5, linestyle='--', label=r'$0.5+\sqrt{\ln k/(2n)}$')
ax.axhline(0.5, color='k', lw=1, linestyle=':')
ax.set_xlabel('number of models scored on the test set, $k$'); ax.set_ylabel('accuracy')
ax.set_title('Choosing on the Test Set Reports Luck as Skill', fontsize=12); ax.legend(fontsize=9); ax.grid(True, alpha=0.3, which='both')
plt.tight_layout(); plt.show()

---
## 3. Problem: One Validation Split Is Noisy

**Problem.** The validation set of Section 1 has about a hundred points. Its score depends on *which* points happened to land in it, so another random split gives another number, and possibly another winner. The points set aside are also lost for fitting.

**Solution: K-fold cross-validation.** Cut the training part into $K$ folds. For each fold, fit on the other $K-1$ and score on that fold; average the $K$ scores. Every point is used for validation exactly once and for fitting $K-1$ times. The price is $K$ fits instead of one. $K=5$ or $10$ is the usual choice; *stratified* folds keep the class proportions in each fold.

**In the cell.** The loop is written out once (`kfold_scores`), checked against `cross_val_score`, and then the same model is scored 100 times with a single split and 100 times with 5-fold CV, changing only the random seed of the split.

In [ ]:
def kfold_scores(model, X, y, K=5, seed=0):
    # K-fold cross-validation written out: fit on K-1 folds, score on the fold left out
    scores = []
    for fit_idx, val_idx in StratifiedKFold(K, shuffle=True, random_state=seed).split(X, y):
        m = clone(model).fit(X[fit_idx], y[fit_idx])
        scores.append(m.score(X[val_idx], y[val_idx]))
    return np.array(scores)

model = make_pipeline(StandardScaler(), SVC(C=1))
ours = kfold_scores(model, X_train, y_train)
lib = cross_val_score(model, X_train, y_train, cv=StratifiedKFold(5, shuffle=True, random_state=0))
print("scores of the 5 folds:", np.round(ours, 3), f" mean = {ours.mean():.3f}")
print("same as cross_val_score:", np.allclose(ours, lib))

single, kfold = [], []
for seed in range(100):
    Xf, Xv, yf, yv = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=seed)
    single.append(clone(model).fit(Xf, yf).score(Xv, yv))
    kfold.append(kfold_scores(model, X_train, y_train, seed=seed).mean())
single, kfold = np.array(single), np.array(kfold)

print(f"\n{'estimate (100 random seeds)':<30} {'mean':>7} {'std':>7} {'min':>7} {'max':>7}")
print(f"{'single 80/20 split':<30} {single.mean():>7.3f} {single.std(ddof=1):>7.3f} {single.min():>7.3f} {single.max():>7.3f}")
print(f"{'5-fold cross-validation':<30} {kfold.mean():>7.3f} {kfold.std(ddof=1):>7.3f} {kfold.min():>7.3f} {kfold.max():>7.3f}")

fig, ax = plt.subplots(figsize=(8, 4.2))
bins = np.linspace(min(single.min(), kfold.min()) - 0.005, 1.005, 30)
ax.hist(single, bins=bins, color=RED, alpha=0.6, label='single 80/20 split')
ax.hist(kfold, bins=bins, color=TEAL, alpha=0.8, label='5-fold cross-validation')
ax.set_xlabel('estimated accuracy of the same model'); ax.set_ylabel('number of seeds (out of 100)')
ax.set_title('Same Model, Same Data: Only the Split Changes', fontsize=12); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 5. Problem: The Winner's Validation Score Is Optimistic Too

**Problem.** Cross-validation is the right tool for *choosing*. But once it has been used to choose, the score of the winner is the best of many noisy scores, which is the situation of Section 2 again, moved from the test set to the validation folds [4]. Reporting the winner's cross-validation score repeats the mistake.

**Solution.** This is the job of the test set: it took no part in the choice, so one evaluation of the chosen model on it is honest. (Without a test set, the alternative is *nested* cross-validation: an outer loop that scores the whole procedure "search + refit" on folds the search never saw.)

**In the cell.** Coin-flip labels once more. 300 candidates are compared by 5-fold CV on the training points; the winner is refitted and scored once on untouched data.

In [ ]:
rng = np.random.default_rng(3)
(X_a, y_a), (X_b, y_b) = coin(200), coin(200)                                  # training points, untouched test points
cv5 = StratifiedKFold(5, shuffle=True, random_state=0)

subsets = [rng.choice(d, 5, replace=False) for _ in range(300)]
cv_acc = np.array([cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=0), X_a[:, f], y_a, cv=cv5).mean() for f in subsets])
winner = int(np.argmax(cv_acc))
final = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_a[:, subsets[winner]], y_a)

print("coin-flip labels: the true accuracy of every model is 0.500\n")
print(f"mean CV accuracy over the 300 candidates : {cv_acc.mean():.3f}")
print(f"CV accuracy of the winner                : {cv_acc[winner]:.3f}   <- used to choose, so not a fair estimate")
print(f"winner on the test set, used once        : {final.score(X_b[:, subsets[winner]], y_b):.3f}   <- the number to report")

---
## 6. The Whole Procedure, From Data to the Reported Number

Now the five steps of Section 0 on the real data. Step 1 was done in the first cell, and the test set has stayed locked since.

**Step 2 — candidates and cross-validation.** Four families from the series, each with a small grid: logistic regression (`07`), RBF SVM (`08`), random forest and gradient boosting (`09`). Each candidate is a `Pipeline`, so the scaler is refitted inside every fold (Section 4). All the candidates are scored on the **same** five stratified folds of the training part (Section 3), which makes the comparison paired.

**Step 3 — the rule, fixed before looking:** *highest mean cross-validation accuracy.* The output also lists the candidates within one standard error of the best. With five folds the ranking among those is not evidence that one is better than another; a common alternative rule, the **one-standard-error rule** [1, §7.10], picks the simplest candidate in that group.

**Step 4 — refit.** The chosen configuration is fitted again on the whole training part: the five models fitted during cross-validation were only there to produce scores, and are discarded.

In [ ]:
K = 5
cv = StratifiedKFold(K, shuffle=True, random_state=0)                      # the same folds for every candidate
scaled = lambda clf: Pipeline([('scale', StandardScaler()), ('clf', clf)])
families = {
    'logistic regression': (scaled(LogisticRegression(max_iter=5000)),      {'clf__C': [0.01, 0.1, 1, 10, 100]}),
    'RBF SVM'            : (scaled(SVC()),                                   {'clf__C': [1, 10, 100], 'clf__gamma': [0.001, 0.01, 0.1]}),
    'random forest'      : (Pipeline([('clf', RandomForestClassifier(200, random_state=0))]), {'clf__max_features': ['sqrt', 0.3]}),
    'gradient boosting'  : (Pipeline([('clf', HistGradientBoostingClassifier(random_state=0))]), {'clf__learning_rate': [0.05, 0.2]}),
}

# Step 2: cross-validate every candidate on the training part only
table = []                                                                   # (family, params, mean, standard error)
for name, (pipe, grid) in families.items():
    search = GridSearchCV(pipe, grid, cv=cv, refit=False).fit(X_train, y_train)
    res = search.cv_results_
    for params, mean, std in zip(res['params'], res['mean_test_score'], res['std_test_score']):
        table.append((name, params, mean, std / np.sqrt(K - 1)))             # std has ddof=0: SE = std / sqrt(K-1)
means, ses = np.array([t[2] for t in table]), np.array([t[3] for t in table])

print(f"{len(table)} candidates, {K}-fold CV on {len(y_train)} training points  ->  {len(table) * K} fits; test set opened: {TEST.opened}\n")
print(f"{'best candidate of each family':<58} {'CV accuracy':>12} {'SE':>7}")
for name in families:
    i = max((j for j, t in enumerate(table) if t[0] == name), key=lambda j: means[j])
    desc = name + '  ' + ', '.join(f"{k.split('__')[1]}={v}" for k, v in table[i][1].items())
    print(f"{desc:<58} {means[i]:>12.4f} {ses[i]:>7.4f}")

# Step 3: the rule fixed in advance
best = int(np.argmax(means))
best_family, best_params = table[best][0], table[best][1]
close = [j for j in range(len(table)) if means[j] >= means[best] - ses[best]]
print(f"\nchosen: {best_family} with {best_params}")
print(f"candidates within one standard error of the best ({means[best] - ses[best]:.4f} or more): {len(close)} of {len(table)}, from the families {sorted({table[j][0] for j in close})}")

# Step 4: refit the chosen candidate on the whole training part
final_model = clone(families[best_family][0]).set_params(**best_params).fit(X_train, y_train)
print(f"\nrefitted on all {len(y_train)} training points; test set opened: {TEST.opened}")

**Step 5 — open the test set.** Every decision has been taken, so the test set is opened here, for the first and only time. The reported number comes with an interval, because the test set is a finite sample: with $n$ test points, one point is worth $1/n$ of accuracy. The cell prints the 95% Wilson interval for a proportion. Since the model was fixed before the test set was opened, the Hoeffding bound for a *single* hypothesis also applies (`02`, §2.2): with probability at least $1-\delta$, the true accuracy is within $\sqrt{\ln(2/\delta)/(2n)}$ of the test accuracy. That guarantee holds for one look only: $k$ looks would bring back the $\ln k$ of Section 2.

The last lines ask for the test set a second time.

In [ ]:
X_test, y_test = TEST.open()                                  # the only call to open() in the notebook
pred = final_model.predict(X_test)
n_test, correct = len(y_test), int(np.sum(pred == y_test))
acc = correct / n_test

z = 1.96                                                      # 95% Wilson interval for a proportion
centre, half = (acc + z**2 / (2*n_test)) / (1 + z**2 / n_test), z * np.sqrt(acc*(1 - acc)/n_test + z**2 / (4*n_test**2)) / (1 + z**2 / n_test)
hoeffding = np.sqrt(np.log(2 / 0.05) / (2 * n_test))

print(f"chosen model            : {best_family}, {best_params}")
print(f"TEST ACCURACY (reported): {acc:.4f}   ({correct} of {n_test} correct; one test point = {1/n_test:.4f})")
print(f"95% Wilson interval     : [{centre - half:.3f}, {centre + half:.3f}]")
print(f"Hoeffding, delta = 0.05 : test accuracy ± {hoeffding:.3f}")
print(f"CV accuracy of the chosen model (Step 3), for comparison: {means[best]:.4f} ± {ses[best]:.4f} (SE)")

cm = np.zeros((2, 2), int)
np.add.at(cm, (y_test, pred), 1)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.8), gridspec_kw={'width_ratios': [2.2, 1]})
ax = axes[0]
colors = dict(zip(families, [BLUE, RED, TEAL, ORANGE]))
order = np.argsort(means)
for pos, j in enumerate(order):
    ax.errorbar(means[j], pos, xerr=ses[j], fmt='o', color=colors[table[j][0]], capsize=3, ms=7 if j == best else 5,
                markeredgecolor='k' if j == best else colors[table[j][0]], markeredgewidth=1.5)
ax.axvline(means[best] - ses[best], color='gray', linestyle='--', lw=1.2, label='best minus one standard error')
for name, c in colors.items():
    ax.plot([], [], 'o', color=c, label=name)
ax.set_yticks([]); ax.set_xlabel(f'{K}-fold CV accuracy on the training part (mean ± SE)')
ax.set_title(f'Step 2-3: {len(table)} Candidates, Chosen One Outlined in Black', fontsize=11); ax.legend(fontsize=8, loc='lower right'); ax.grid(True, alpha=0.3, axis='x')
ax = axes[1]
ax.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=14, color='white' if cm[i, j] > cm.max() / 2 else 'black')
ax.set_xticks([0, 1]); ax.set_yticks([0, 1]); ax.set_xticklabels(['malignant', 'benign']); ax.set_yticklabels(['malignant', 'benign'])
ax.set_xlabel('predicted'); ax.set_ylabel('true'); ax.set_title(f'Step 5: Test Set, Used Once\naccuracy {acc:.3f}', fontsize=11)
plt.tight_layout(); plt.show()

# A second request is refused
try:
    TEST.open()
except RuntimeError as err:
    print("second TEST.open():", err)

---
## 7. Checklist and Summary

**What to do with the reported number.** It is reported as it is, whether or not it is pleasant. If it disappoints and the model is changed in response, the test set has become a validation set, and an honest estimate needs **new** test data. The model that is finally deployed may be refitted on training plus test, since more data usually helps, but then no score for that last model is available: the reported number is the estimate.

| Problem | Symptom | Solution |
|---|---|---|
| Scoring on the training data (§1) | Training accuracy near 100%, poor on new data | Score on data not used for fitting |
| Choosing on the test set (§2) | Reported score above the real one, by roughly $\sqrt{\ln k/(2n)}$ after $k$ looks | Choose on validation data; open the test set once |
| A single validation split (§3) | Score and winner change with the random seed | K-fold cross-validation, stratified, same folds for all candidates |
| Preprocessing fitted before the split (§4) | High cross-validation score, even on noise | `Pipeline`: every fitted step inside the fold |
| Reporting the winner's CV score (§5) | Optimistic, because it was used to choose | Report the test score (or nested cross-validation) |
| A score without uncertainty (§6) | Differences of one or two test points read as real | Report $n$ and an interval; compare with the standard error |

**Before reporting a number, check:**

1. The test set was separated before anything else, including exploratory plots and preprocessing.
2. Every choice was made by cross-validation on the training part.
3. Everything fitted to data is inside the pipeline that was cross-validated.
4. The selection rule was fixed before the scores were seen.
5. The chosen model was refitted on the whole training part.
6. The test set was used once, and the result is reported with its interval.

Two cases need a different way of splitting, which is not covered here: **grouped data** (several rows from the same patient or user must stay in the same fold) and **time series** (validate on the future, never on the past). In both, a random split is itself a form of leakage [5].

---

**End of the series.** Notebooks `01`–`02` explained *why* a model fitted on a sample can work on new data, `03`–`09` built the models, and this notebook fixed the procedure that makes the final number worth believing.